In [1]:
!pip install neuralforecast -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 78.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.


In [2]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 28.5 MB/s eta 0:00:00


In [3]:
!pip install workalendar -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 122.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.9 MB/s eta 0:00:00


In [4]:
!pip install optuna -q

In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from workalendar.europe import Russia
from tqdm import tqdm
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
df = pd.read_csv("df_final+whether.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

horizons = {
    "4h":  8,
    "8h":  16,
    "24h": 48,
    "7d":  336,
    "14d": 672,
    "1m":  1488,
}

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
input_window = 336

In [7]:
class TimeSeriesDatasetMV(Dataset):
    def __init__(self, data, input_window, horizon):
        self.data = data
        self.input_window = input_window
        self.horizon = horizon

    def __len__(self):
        return len(self.data) - self.input_window - self.horizon + 1

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.input_window]
        y = self.data[idx + self.input_window: idx + self.input_window + self.horizon, 0]
        return torch.FloatTensor(x), torch.FloatTensor(y)

In [8]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=7, hidden_size=64, num_layers=2, horizon=48, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])

In [9]:
scalers = {}

def prepare_data(house):
    scaler = MinMaxScaler()
    features = pd.DataFrame({
        "power": scaler.fit_transform(df[[house]]).flatten(),
        "hour": df["timestamp"].dt.hour.values / 23.0,
        "weekday": df["timestamp"].dt.weekday.values / 6.0,
        "month": df["timestamp"].dt.month.values / 12.0,
        "temp_c": (df["temp_c"] - df["temp_c"].min()) / (df["temp_c"].max() - df["temp_c"].min()),
        "is_weekend": (df["timestamp"].dt.weekday >= 5).astype(float).values,
        "is_holiday": df["is_holiday"].astype(float).values,
    })
    scalers[house] = scaler
    data = features.values.astype(np.float32)
    return data[:train_end], data[train_end:val_end], data[val_end:], scaler

def train_lstm(model, loader_train, loader_val, epochs=100, lr=0.001, patience=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_val_loss = float("inf")
    patience_counter = 0
    best_weights = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x_batch, y_batch in loader_train:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in loader_val:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                val_loss += criterion(model(x_batch), y_batch).item()

        train_loss /= len(loader_train)
        val_loss /= len(loader_val)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_weights)
    return model

def predict_lstm(model, val_data, test_data, input_window, horizon, scaler):
    model.eval()
    predictions, actuals = [], []
    context = np.concatenate([val_data, test_data])

    with torch.no_grad():
        for i in range(0, len(context) - input_window - horizon + 1, horizon):
            x = context[i: i + input_window]
            y = context[i + input_window: i + input_window + horizon, 0]
            if len(y) < horizon:
                break
            x_tensor = torch.FloatTensor(x).unsqueeze(0).to(device)
            y_pred = model(x_tensor).cpu().numpy().flatten()
            predictions.extend(y_pred)
            actuals.extend(y)

    predictions = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
    actuals = scaler.inverse_transform(np.array(actuals).reshape(-1, 1)).flatten()
    return actuals, predictions

In [10]:
# подбор гиперпараметров через Optuna (на house_1, горизонт 24h)
def objective(trial, house, horizon_points):
    hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    train_data, val_data, test_data, scaler = prepare_data(house)

    dataset_train = TimeSeriesDatasetMV(train_data, input_window, horizon_points)
    dataset_val = TimeSeriesDatasetMV(val_data, input_window, horizon_points)
    loader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
    loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)

    model = LSTMModel(
        input_size=7,
        hidden_size=hidden_size,
        num_layers=num_layers,
        horizon=horizon_points,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(50):
        model.train()
        for x_batch, y_batch in loader_train:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in loader_val:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                val_loss += criterion(model(x_batch), y_batch).item()
        val_loss /= len(loader_val)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= 5:
            break

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_loss

print("Подбор гиперпараметров (house_1, горизонт 24h)...")
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study.optimize(
    lambda trial: objective(trial, "house_1", horizons["24h"]),
    n_trials=20,
    show_progress_bar=True
)

best_params = study.best_params
print(f"\nЛучшие параметры:")
for key, value in best_params.items():
    print(f"  {key}: {value}")
print(f"Лучший val_loss: {study.best_value:.6f}")

Подбор гиперпараметров (house_1, горизонт 24h)...


  0%|          | 0/20 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.12357135994665616 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.17810372945986974 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1958501260498623 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropo


Лучшие параметры:
  hidden_size: 128
  num_layers: 2
  dropout: 0.23014646249853477
  lr: 0.002308096943325736
  batch_size: 16
Лучший val_loss: 0.001013


In [11]:
# обучение на всех домах и горизонтах с лучшими параметрами
lstm_results = {}

for house in tqdm(houses, desc="LSTM дома"):
    train_data, val_data, test_data, scaler = prepare_data(house)
    house_results = {}

    for horizon_name, horizon_points in tqdm(
        horizons.items(),
        desc=f"  {house}",
        leave=False
    ):
        dataset_train = TimeSeriesDatasetMV(train_data, input_window, horizon_points)
        dataset_val = TimeSeriesDatasetMV(val_data, input_window, horizon_points)
        loader_train = DataLoader(dataset_train, batch_size=best_params["batch_size"], shuffle=True)
        loader_val = DataLoader(dataset_val, batch_size=best_params["batch_size"], shuffle=False)

        model = LSTMModel(
            input_size=7,
            hidden_size=best_params["hidden_size"],
            num_layers=best_params["num_layers"],
            horizon=horizon_points,
            dropout=best_params["dropout"]
        ).to(device)

        model = train_lstm(
            model, loader_train, loader_val,
            epochs=100, lr=best_params["lr"], patience=10
        )

        actuals, predictions = predict_lstm(
            model, val_data, test_data, input_window, horizon_points, scaler
        )
        metrics = evaluate(actuals, predictions)
        house_results[horizon_name] = metrics

        tqdm.write(f"{house} {horizon_name}: MAE={metrics['MAE']:.3f}, MAPE={metrics['MAPE']:.3f}%")

        torch.save(model.state_dict(), f"lstm_{house}_{horizon_name}.pth")

    lstm_results[house] = house_results

# сводная таблица
rows = []
for house in houses:
    for horizon_name, metrics in lstm_results[house].items():
        rows.append({
            "модель": "LSTM",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
        })

df_lstm = pd.DataFrame(rows)

for metric in ["MAE", "RMSE", "MAPE"]:
    pivot = df_lstm.pivot(index="горизонт", columns="дом", values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    print(f"\nLSTM - {metric}:")
    print(pivot.to_string())

df_lstm.to_csv("results_lstm.csv", index=False)

LSTM дома:   0%|          | 0/8 [00:00<?, ?it/s]

  house_1:  17%|█▋        | 1/6 [02:52<14:20, 172.19s/it]

house_1 4h: MAE=2.679, MAPE=5.016%



  house_1:  33%|███▎      | 2/6 [05:35<11:09, 167.25s/it]

house_1 8h: MAE=2.679, MAPE=5.027%



  house_1:  50%|█████     | 3/6 [08:26<08:25, 168.66s/it]

...

house_8 7d: MAE=2.250, MAPE=6.698%



  house_8:  83%|████████▎ | 5/6 [14:37<02:39, 159.39s/it]

house_8 14d: MAE=2.323, MAPE=6.927%



LSTM дома: 100%|██████████| 8/8 [2:11:37<00:00, 987.17s/it]

house_8 1m: MAE=2.437, MAPE=7.274%

LSTM - MAE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
горизонт                                                                        
4h          2.679    1.430    1.057    1.939    1.481    3.920    3.416    1.672
8h          2.679    1.497    1.105    2.048    1.460    4.288    3.651    1.791
24h         3.052    1.596    1.186    2.202    1.622    4.836    3.981    2.070
7d          3.836    1.772    1.349    2.435    1.733    6.071    5.087    2.250
14d         4.186    1.880    1.438    2.451    1.811    5.836    5.503    2.323
1m          4.301    1.912    1.351    2.612    1.784    6.084    5.938    2.437

LSTM - RMSE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
горизонт                                                                        
4h          3.587    1.916    1.388    2.570    1.986    5.275    4.495    2.238
8h          3.675    2.017    1.448    2.710   

In [12]:
# загружаем лучшую модель для house_1, горизонт 24h
train_data, val_data, test_data, scaler = prepare_data("house_1")
horizon_points = horizons["24h"]

model_plot = LSTMModel(
    input_size=7,
    hidden_size=best_params["hidden_size"],
    num_layers=best_params["num_layers"],
    horizon=horizon_points,
    dropout=best_params["dropout"]
).to(device)
model_plot.load_state_dict(torch.load("lstm_house_1_24h.pth"))

actuals, predictions = predict_lstm(
    model_plot, val_data, test_data, input_window, horizon_points, scaler
)

timestamps = df["timestamp"].iloc[val_end:val_end + horizon_points]

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    model_h = LSTMModel(
        input_size=7,
        hidden_size=best_params["hidden_size"],
        num_layers=best_params["num_layers"],
        horizon=horizon_points,
        dropout=best_params["dropout"]
    ).to(device)
    model_h.load_state_dict(torch.load(f"lstm_house_1_{horizon_name}.pth"))

    actuals_h, predictions_h = predict_lstm(
        model_h, val_data, test_data, input_window, horizon_points, scaler
    )

    timestamps_h = df["timestamp"].iloc[val_end:val_end + len(actuals_h)]

    fig.add_trace(go.Scatter(
        x=timestamps_h[:horizon_points],
        y=actuals_h[:horizon_points],
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=timestamps_h[:horizon_points],
        y=predictions_h[:horizon_points],
        mode="lines", name="прогноз LSTM",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text="Дата", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="кВт", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="LSTM: Прогнозные и факктические значения по всем горизонтамам прогнозирования (дом 1)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image("33_lstm_forecast_all_horizons.png", scale=2)